# 04 — Held-out source-video evaluation

For each fold and seed, the encoder and every fitted read-out see only
outer-training sources. Ridge selection uses source-disjoint inner folds;
scaling, fitting, neutral-band calibration, and target scale are train-only.
Predictions for original and mirrored poses are then produced once on the
held-out sources with the same fitted read-out.

The lane factorial separates single-pass free prediction, two-pass odd/even
parity, and the zero-origin read-out constraint for both learned and paired
initial encoders. Measured visibility, acquisition, annotation, and combined
nuisance lanes are accompanied by a learned-plus-nuisance incremental lane.
The target-component oracle is self-consistency only, never a baseline.

Before any read-out, the notebook also compares `E(Mx)` with `S E(x)` for all
33 joints on exactly common-valid tokens. Its strict error is residual energy
divided by total representation energy: zero is exact, while unrelated
equal-energy representations are near one. `S` swaps joints but does not fit,
rotate, sign-flip, or align latent channels. This conservative diagnostic is
evaluated per checkpoint against its paired initialization. Mirrors remain
paired transformations, not new test cases.
Results remain about source videos, while folder labels remain annotations and
no clinical or unseen-person interpretation is permitted. Synthetic smoke
scores are pipeline diagnostics, not evidence.

## What this notebook evaluates

Notebook 03 produced encoders; notebook 04 asks how each frozen encoder behaves
on source videos that were sealed away from its training fold. Nothing here
updates encoder weights. A paper-profile **evaluation job** is one training
variant, one outer fold, and one optimization seed. Five folds, five seeds, and
two variants therefore give $5	imes5	imes2=50$ evaluation jobs.

Within a job, the outer-test targets remain untouched while the read-out recipe
is chosen. Seven **ridge regression** penalties are compared using the four inner
folds made only from outer-training sources. Ridge regression is a linear
prediction rule whose penalty discourages unstable, excessively large
coefficients. After the penalty is selected, the read-out is refitted on all
outer-training sources and used once on the held-out sources. Scaling,
missing-value imputation, the neutral band, and target scale are also calculated
from training sources only.

The 16 registered feature **lanes** answer deliberately different questions:

- the primary `learned_single_free` lane asks what a single encoder pass plus an
  unconstrained linear read-out can predict;
- learned and paired-initialization two-pass lanes separate odd and even content;
- zero-origin odd lanes impose exact sign reversal and are construction controls;
- visibility, acquisition, annotation, and combined nuisance lanes test measured
  non-representation explanations;
- the learned-plus-nuisance lane asks whether learned features add information
  beyond those measured controls; and
- the target-component oracle reconstructs a target from its own ingredients. It
  checks self-consistency and is never a fair predictive baseline.

Before fitting any read-out, the strict token audit compares $E(Mx)$ with
$S E(x)$. Here $E$ is the encoder, $x$ is a pose sequence, $M$ anatomically
mirrors the coordinates, and $S$ swaps the corresponding left/right token
joints without fitting an alignment. The error is squared residual energy
divided by total representation energy. Zero means exact agreement; lower is
better. The calculation uses only tokens valid in both paths.

<svg viewBox="0 0 1080 340" width="100%" role="img"
     aria-labelledby="evaluation-flow-title evaluation-flow-description"
     xmlns="http://www.w3.org/2000/svg">
  <title id="evaluation-flow-title">Held-out evaluation workflow for one checkpoint</title>
  <desc id="evaluation-flow-description">One fold-local checkpoint encodes
  outer-training and sealed outer-test poses. Inner training folds select ridge
  penalties for sixteen read-out lanes. The final read-outs predict original and
  mirrored held-out poses, while a direct token audit compares the learned and
  paired-initial encoders.</desc>
  <defs><marker id="arrow04" markerWidth="8" markerHeight="8" refX="7"
    refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#475569"/></marker></defs>
  <style>
    .box04{fill:#f8fafc;stroke:#334155;stroke-width:1.5}
    .train04{fill:#ecfdf5;stroke:#047857;stroke-width:1.5}
    .test04{fill:#fff7ed;stroke:#c2410c;stroke-width:1.6}
    .audit04{fill:#eff6ff;stroke:#2563eb;stroke-width:1.5}
    .line04{stroke:#475569;stroke-width:1.8;fill:none;marker-end:url(#arrow04)}
    .h04{font:600 14px system-ui,sans-serif;fill:#0f172a}
    .s04{font:12px system-ui,sans-serif;fill:#475569}
  </style>
  <rect class="box04" x="15" y="125" width="180" height="84" rx="9"/>
  <text class="h04" x="105" y="154" text-anchor="middle">Frozen checkpoint</text>
  <text class="s04" x="105" y="177" text-anchor="middle">variant + fold + seed</text>
  <text class="s04" x="105" y="197" text-anchor="middle">encoder weights do not change</text>
  <rect class="train04" x="250" y="35" width="210" height="105" rx="9"/>
  <text class="h04" x="355" y="65" text-anchor="middle">Outer-training sources</text>
  <text class="s04" x="355" y="88" text-anchor="middle">four inner source folds</text>
  <text class="s04" x="355" y="108" text-anchor="middle">select ridge penalty</text>
  <text class="s04" x="355" y="128" text-anchor="middle">fit scaler + read-out</text>
  <rect class="test04" x="250" y="215" width="210" height="105" rx="9"/>
  <text class="h04" x="355" y="245" text-anchor="middle">Sealed outer-test sources</text>
  <text class="s04" x="355" y="268" text-anchor="middle">original pose x</text>
  <text class="s04" x="355" y="288" text-anchor="middle">anatomical mirror Mx</text>
  <text class="s04" x="355" y="308" text-anchor="middle">targets never tune the read-out</text>
  <path class="line04" d="M195 151 C220 151 220 87 250 87"/>
  <path class="line04" d="M195 183 C220 183 220 267 250 267"/>
  <rect class="box04" x="525" y="35" width="225" height="105" rx="9"/>
  <text class="h04" x="637" y="65" text-anchor="middle">Sixteen registered lanes</text>
  <text class="s04" x="637" y="88" text-anchor="middle">learned + initialization floors</text>
  <text class="s04" x="637" y="108" text-anchor="middle">odd/even + nuisance controls</text>
  <text class="s04" x="637" y="128" text-anchor="middle">same train-only tuning rule</text>
  <path class="line04" d="M460 87 L525 87"/>
  <rect class="audit04" x="525" y="215" width="225" height="105" rx="9"/>
  <text class="h04" x="637" y="245" text-anchor="middle">Direct token audit</text>
  <text class="s04" x="637" y="268" text-anchor="middle">compare E(Mx) with S E(x)</text>
  <text class="s04" x="637" y="288" text-anchor="middle">learned versus paired initial</text>
  <text class="s04" x="637" y="308" text-anchor="middle">common-valid tokens only</text>
  <path class="line04" d="M460 267 L525 267"/>
  <rect class="audit04" x="815" y="125" width="245" height="84" rx="9"/>
  <text class="h04" x="937" y="154" text-anchor="middle">Saved held-out rows</text>
  <text class="s04" x="937" y="177" text-anchor="middle">prediction + mirror prediction</text>
  <text class="s04" x="937" y="197" text-anchor="middle">strict token errors + lineage</text>
  <path class="line04" d="M750 87 C785 87 785 151 815 151"/>
  <path class="line04" d="M750 267 C785 267 785 183 815 183"/>
</svg>

## How to read the live evaluation progress display

The overall bar counts evaluation jobs, not sequences or read-out lanes. The
active label names the variant, outer fold, and seed. A fresh job performs four
expensive encoder passes—learned and paired-initial encoders on original and
mirrored poses—then tunes all 16 lanes. The display updates after each job, so
a single job can remain active for a while without implying that execution has
stalled.

A saved comma-separated values (CSV) file is only a **cached candidate**. It
counts as reused after the checkpoint lineage, file fingerprint, row coverage,
source weights, and result digest all validate. A mismatch stops the run rather
than silently recomputing into a questionable directory. A fresh result is
written atomically, meaning an incomplete file does not replace a valid result.

Estimated time of arrival (ETA) appears after the first newly computed job and
uses the median duration of completed new jobs. Cache validation time is not used
to predict fresh computation. Source folds contain different sequence counts,
and CPU, GPU, or Apple Metal Performance Shaders (MPS) load can vary, so the
estimate is a guide rather than a deadline. Re-running the cell safely validates
and reuses completed jobs; an interrupted active job is the only work that may
need to be repeated.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from laterality.data import load_cohort
from laterality.splitting import load_splits
from laterality.visualization import evaluation_figure
from notebook_progress import (
    NotebookTaskProgress,
    evaluate_selected_with_progress,
)

cohort = load_cohort(context)
splits = load_splits(context, cohort)
evaluation_progress = NotebookTaskProgress(
    "Held-out evaluation progress",
    "job",
    refresh_seconds=1.0,
)
evaluations = evaluate_selected_with_progress(
    context,
    cohort,
    splits,
    progress=evaluation_progress,
)

evaluation_summary = (
    evaluations.groupby(["variant", "fold", "seed", "lane"], as_index=False)
    .agg(test_sequences=("sequence_id", "size"), test_sources=("video_id", "nunique"))
    .sort_values(["variant", "fold", "seed", "lane"])
    .reset_index(drop=True)
)
representation_summary = (
    evaluations[
        evaluations["lane"]
        == context.protocol["evaluation"]["primary_lane"]
    ]
    .groupby(["variant", "seed"], as_index=False)
    .agg(
        learned_strict_error=(
            "learned_strict_equivariance_error",
            "mean",
        ),
        initial_strict_error=(
            "initial_strict_equivariance_error",
            "mean",
        ),
        minimum_common_tokens=(
            "learned_strict_equivariance_common_tokens",
            "min",
        ),
    )
    .sort_values(["variant", "seed"])
    .reset_index(drop=True)
)
display(evaluation_summary)
display(representation_summary)
show_inline(evaluation_figure(context, evaluations))

## Step-by-step interpretation of the evaluation output

### 1. Verify completeness before interpreting magnitudes

In a complete paper run, `evaluation_summary` has 800 rows: 50 checkpoint jobs
times 16 lanes. Within a particular fold, every lane must report the same held-out
source and sequence counts because lanes are alternative feature/read-out recipes
applied to the same test observations. Expected source counts are 19, 19, 19, 18,
and 18 for folds 0 through 4. Expected sequence counts are 189, 182, 72, 77, and
105. A missing job, lane, source, or sequence is an integrity failure—not a zero
score and not something aggregation should ignore.

The table does not yet report prediction quality. Its job is to prove coverage:
`variant` names the training recipe, `fold` names the held-out source partition,
`seed` names the optimization repeat, and `lane` names the feature/read-out
recipe. Repeated test counts across lanes are expected rather than duplication.

### 2. Read the representation summary as a diagnostic

`representation_summary` has one row for each variant/seed combination after
pooling the five outer folds, normally ten rows. `learned_strict_error` measures
the trained encoder, while `initial_strict_error` measures that checkpoint's
paired pre-training initialization. Lower values mean closer agreement with the
fixed anatomical token swap. `minimum_common_tokens` is the smallest number of
jointly valid tokens supporting any included comparison; it must remain at or
above the registered minimum of eight.

This compact printed summary uses an ordinary mean over sequence rows and is
descriptive. The plotted comparisons and notebook-05 inference balance sources
so a video that yielded many clips does not dominate the scientific conclusion.

### 3. Interpret both panels of the figure

In the left panel, the initial and learned bars summarize strict token error for
each variant. The dashed line is the registered absolute error margin of 0.1.
A learned bar below its initial bar suggests improvement, and a learned bar below
the dashed line suggests small absolute error. Neither visual suggestion alone is
a passed claim: notebook 05 requires source-bootstrap confidence bounds for both
the absolute margin and learned-minus-initial improvement.

In the right panel, each dot is a paired source/checkpoint comparison. The
horizontal coordinate is initial-encoder error and the vertical coordinate is
learned-encoder error. A dot below the diagonal favors the learned encoder; a dot
above it favors the initialization. For example, `(0.30, 0.12)` indicates a large
reduction but still misses the 0.1 absolute margin, whereas `(0.08, 0.07)` meets
the point margin but shows little training improvement. The registered conclusion
needs both kinds of evidence with uncertainty included.

### 4. Keep prediction and symmetry questions separate

A representation can transform cleanly yet carry too little information to
predict the coordinate-derived target. Conversely, a read-out can predict the
target while internal tokens fail the strict swap test. That is why this notebook
stores predictions, mirrored predictions, and direct token errors separately.
Exact oddness in a zero-origin two-pass lane is imposed by construction; it does
not prove that the single-pass encoder learned equivariance.

### 5. Interpret the current paper-profile findings

The completed paper evaluation contains all 50 jobs, all 16 lanes, and 100,000
saved rows. Across the five outer folds, each of the 625 accepted sequences and
each of the 93 source videos is tested exactly once for every variant/seed
combination. The minimum common-token count is 196, comfortably above the
registered minimum of eight, so these strict-error values are not being driven by
the minimum-support cutoff.

After giving each source equal total influence, the learned strict-error means are
0.1053 for reflection augmentation and 0.1138 for vanilla, compared with 0.0832
for their matched initial encoders. Lower is better, so training did not improve
the descriptive mean: learned error is higher than initial error for every seed in
both variants. The five learned seed means range from 0.0831 to 0.1233 for
reflection augmentation and from 0.0845 to 0.1324 for vanilla. Only two augmented
seed means and one vanilla seed mean fall below 0.1.

The source-level picture is mixed rather than uniform. Learned error is lower than
paired-initial error in 220 of 465 source/seed comparisons (47.3%) for reflection
augmentation and 192 of 465 (41.3%) for vanilla. A few sources have much larger
errors, which is why the means exceed the medians and why source-level uncertainty
matters. Reflection augmentation has the lower descriptive learned mean by about
0.0084, but that point difference is not yet a supported augmentation effect.

The narrow descriptive reading is therefore cautious: **the direct token audit
currently gives no visible evidence that training improved strict anatomical-swap
equivariance over initialization, and several learned seed means miss the 0.1
absolute margin.** Notebook 05 must still compute paired source-bootstrap
confidence bounds before the registered gate receives its final `SUPPORTED` or
`NOT SUPPORTED` status. These token results also say nothing yet about held-out
target predictability.

### 6. State only the handoff conclusion

A complete notebook-04 run establishes that all registered checkpoints were
evaluated on their untouched source folds with train-only read-out selection and
that the paired mirror/token diagnostics were saved. It still does not establish
positive predictive utility, a statistically supported symmetry effect, external
generalization, unseen-person performance, or a clinical result. Notebook 05
performs the source-balanced aggregation and applies the locked decision rules.